In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('products_asos.csv', on_bad_lines='skip', engine='python')

print(f"Initial Data Loaded: {len(df)} rows")

df['price'] = pd.to_numeric(df['price'], errors='coerce')
df = df.dropna(subset=['price'])

print(f"Data after dropping rows with missing price: {len(df)} rows")

df.head()

In [2]:
df['description'] = df['description'].astype(str)

def get_brand(text):
  if 'by' in text:
    try:
       return text.split('by ')[1].split(' ')[0]
    except:
        return "Unknown"
  return "Unknown"


In [4]:
brand_map = {
    'New': 'New Look',
    'River': 'River Island',
    'Miss': 'Miss Selfridge',
    'Under': 'Under Armour',
    'Vero': 'Vero Moda',
    'In': 'In The Style',
    'River': 'River Island',
}


In [5]:
import re

def split_camel_case(brand):
    if not isinstance(brand, str):
        return brand

    # Split at capital letter (StradivariusThrow → Stradivarius)
    match = re.match(r"([A-Z][a-z]+)", brand)
    if match:
        return match.group(1)

    return brand

In [6]:
df['brand_raw'] = df['description'].apply(get_brand)

# FIX glued brands
df['brand_raw'] = df['brand_raw'].apply(split_camel_case)

df['Brand'] = df['brand_raw'].map(brand_map).fillna(df['brand_raw'])

In [ ]:
brand_counts = df['Brand'].value_counts()
valid_brands = brand_counts[brand_counts > 5].index
df_clean = df[df['Brand'].isin(valid_brands)].copy()

print(df_clean['Brand'].value_counts().head(5))

In [ ]:
 # Function to analyze stockouts
def calculate_stockout_metrics(size_str):
  if not isinstance(size_str, str):
    return 0, 0.0

  # split "UK 6, UK 8 - out of stock" into list
  sizes = size_str.split(',')
  total_sizes = len(sizes)

  #count how many items are out of stock
  out_of_stock_count = size_str.count('Out of stock')

  #calculate rate (0.0 to 1.0)
  rate = out_of_stock_count / total_sizes if total_sizes > 0 else 0.0
  return out_of_stock_count, rate


metrics = df_clean['size'].apply(lambda x:calculate_stockout_metrics(x))

df_clean['Stockout_count'] = [x[0] for x in metrics]
df_clean['Stockout_rate'] = [x[1] for x in metrics]

df_clean['Lost_revenue'] = df_clean['price'] * df_clean['Stockout_count']

cols = ['Brand', 'name', 'price', 'Stockout_count', 'Stockout_rate', 'Lost_revenue']
print(df_clean[cols].sort_values(by='Lost_revenue', ascending=False).head(5))


In [ ]:
brand_strategy = df_clean.groupby('Brand').agg({
    'price': 'mean',
    'Stockout_rate': 'mean',
    'Lost_revenue': 'sum',
    'name': 'count'
}).reset_index()

brand_strategy = brand_strategy[brand_strategy['name'] > 10]

plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=brand_strategy,
    x='price',
    y='Stockout_rate',
    size='Lost_revenue',
    sizes=(50, 500),
    alpha=0.7,
    palette='viridis'
)

winners = brand_strategy[
    (brand_strategy['price']> 40) &
    (brand_strategy['Stockout_rate']>0.4)
]

for i in range(len(winners)):
  plt.text(
      winners.iloc[i]['price']+1,
      winners.iloc[i]['Stockout_rate'],
      winners.iloc[i]['Brand'],
  )

plt.title('Brand Strategy Analysis')
plt.xlabel('Average Price')
plt.ylabel('Stockout Rate')
plt.axvline(x=40, color='red', linestyle='--')
plt.axhline(y=0.4, color='red', linestyle='--')
plt.show()

